In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [0]:
spark = SparkSession.builder.appName("employee").getOrCreate()

In [0]:
data = [
    (1, "Rahul", "IT", "2024-01-01", "Present"),
    (2, "Priya", "HR", "2024-01-01", "Absent"),
    (1, "Rahul", "IT", "2024-01-02", "Present"),
    (3, "Amit", "IT", "2024-01-01", "Present"),
    (2, "Priya", "HR", "2024-01-02", "Present"),
    (4, "Neha", "Finance", "2024-01-01", "Present")
]

columns = ["emp_id", "emp_name", "dept", "log_date", "status"]

df = spark.createDataFrame(data, columns)

In [0]:
display(df)

In [0]:
from pyspark.sql.window import Window

In [0]:
windowSpec = Window.partitionBy("dept").orderBy("log_date")

present_status = df.filter(F.col("status")=='Present').groupBy("emp_id").agg(F.count("status").alias("total_present_day"))
final_df = df.join(present_status, on='emp_id', how='inner').withColumn("rank", F.rank().over(windowSpec)).filter(F.col("rank")==1).select("emp_id", "emp_name", "dept", "total_present_day").orderBy(F.desc(F.col("total_present_day")))
display(final_df)

In [0]:
display(df)

In [0]:
df1 = df.withColumn("present_flag",(F.col("status")=="Present").cast("int"))
display(df1)

In [0]:
df2 = df1.groupBy("emp_name","emp_id","dept").agg(F.sum("present_flag").alias("total_present"))
display(df2)

In [0]:
#Find employee higest attendance
windowSpec = Window.partitionBy("dept").orderBy(F.desc("total_present"))
final_df = df2.withColumn("rank",F.row_number().over(windowSpec))
final_df = final_df.filter(F.col("rank")==1).drop("rank")
display(final_df)